# Probability Calibration: Isotonic Regression

This notebook demonstrates Isotonic Regression, a non-parametric calibration method that learns a monotonic transformation of model outputs to calibrated probabilities.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss
from sklearn.metrics import calibration_curve

np.random.seed(42)

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target

# Binary classification: Setosa vs. others
y_binary = (y == 0).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.3, random_state=42, stratify=y_binary
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

In [ ]:
svm = SVC(kernel='rbf', random_state=42, probability=True)
svm.fit(X_train, y_train)
proba_uncal = svm.predict_proba(X_test)

y_pred = svm.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"Uncalibrated SVM Accuracy: {acc:.4f}")

In [ ]:
# Isotonic regression calibration
calibrator_iso = CalibratedClassifierCV(SVC(kernel='rbf', random_state=42, probability=True),
                                        method='isotonic', cv=5)
calibrator_iso.fit(X_train, y_train)

proba_iso = calibrator_iso.predict_proba(X_test)
y_pred_iso = calibrator_iso.predict(X_test)
acc_iso = accuracy_score(y_test, y_pred_iso)

print(f"Isotonic-calibrated SVM Accuracy: {acc_iso:.4f}")

In [ ]:
# Platt scaling for comparison
calibrator_platt = CalibratedClassifierCV(SVC(kernel='rbf', random_state=42, probability=True),
                                          method='sigmoid', cv=5)
calibrator_platt.fit(X_train, y_train)
proba_platt = calibrator_platt.predict_proba(X_test)

# Evaluate both
brier_uncal = brier_score_loss(y_test, proba_uncal[:, 1])
brier_iso = brier_score_loss(y_test, proba_iso[:, 1])
brier_platt = brier_score_loss(y_test, proba_platt[:, 1])

print("\n=== BRIER SCORE COMPARISON (lower is better) ===")
print(f"Uncalibrated: {brier_uncal:.4f}")
print(f"Platt Scaling: {brier_platt:.4f}")
print(f"Isotonic Regression: {brier_iso:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

methods = [('Uncalibrated', proba_uncal), ('Platt Scaling', proba_platt), ('Isotonic Regression', proba_iso)]

for ax, (name, proba) in zip(axes, methods):
    frac_pos, mean_pred = calibration_curve(y_test, proba[:, 1], n_bins=10)
    ax.plot(mean_pred, frac_pos, 'o-', linewidth=2, markersize=8, label=name)
    ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction of positives')
    ax.set_title(name)
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

prob_list = [('Uncalibrated', proba_uncal), ('Platt Scaling', proba_platt), ('Isotonic Regression', proba_iso)]

for ax, (name, proba) in zip(axes, prob_list):
    ax.hist(proba[y_test==0, 1], bins=20, alpha=0.7, label='Negative', color='blue')
    ax.hist(proba[y_test==1, 1], bins=20, alpha=0.7, label='Positive', color='red')
    ax.set_xlabel('Probability')
    ax.set_ylabel('Frequency')
    ax.set_title(name)
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
comparison = {
    'Aspect': ['Parametric', 'Flexibility', 'Data efficiency', 'Monotonicity', 'Best for'],
    'Platt Scaling': ['Yes (2 params)', 'Low', 'Efficient', 'Assumed smooth', 'Near-sigmoid outputs'],
    'Isotonic Regression': ['No (data-driven)', 'High', 'Needs more data', 'Enforced', 'Complex patterns']
}

import pandas as pd
df_comp = pd.DataFrame(comparison)
print("\n=== CALIBRATION METHOD COMPARISON ===")
print(df_comp.to_string(index=False))

In [ ]:
print("""
Isotonic Regression Calibration:
- Non-parametric: learns arbitrary monotonic mapping
- More flexible than Platt scaling
- Requires more data to avoid overfitting
- Always maintains monotonicity (good for probabilities)
- Can handle complex uncalibrated patterns

When to use:
- Isotonic if you have enough calibration data and want flexibility
- Platt if you have limited data or outputs look sigmoid-shaped
""")